# Reproduce the paper's figures and tables

Each figure and table is built by its own script in `analysis/`, reading the saved results (`frozen_release_v2/`, `results/`, `splits/`). Run the cells in order. Outputs are written to `outputs/` (figures as PDF + PNG, tables as CSV + LaTeX rows).

In [ ]:
# 1. Get the repository with the analysis scripts (safe to re-run)
import os, sys, glob, subprocess, importlib
REPO, BRANCH = '/content/fet', 'release-v2-validity-audit'
def git(*a): subprocess.run(['git', *a], cwd=REPO, check=True)

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/isah-pentest/forensic-email-triage.git', REPO], check=True)
# Until the branch is on GitHub, upload forensic-email-triage-release-v2.bundle to /content (newest upload is used).
bundles = sorted(glob.glob('/content/*.bundle'), key=os.path.getmtime)
source = bundles[-1] if bundles else 'origin'
git('fetch', '-q', source, BRANCH)                 # fetch into FETCH_HEAD, then (re)point the branch at it
git('checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD')
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO, capture_output=True, text=True).stdout, 'from', source)
assert os.path.isdir(f'{REPO}/analysis'), 'analysis/ missing: upload the latest bundle to /content and re-run this cell'
os.chdir(REPO)
if REPO not in sys.path: sys.path.insert(0, REPO)
importlib.invalidate_caches()

In [ ]:
# 2. (Optional) read the saved results from Google Drive instead of the repository copy
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    os.environ['FET_V2'] = '/content/drive/MyDrive/forensic_email_project/frozen_release_v2'
from analysis import common as C
C.paths()

## Figure 1 — learning curve (source: `E3_nested_learning_curve_tfidf.csv`)

In [ ]:
from analysis import fig1_learning_curve
fig1_learning_curve.make(show=True)

## Figure 2 — review budget (source: 100k text-model prediction files)

In [ ]:
from analysis import fig2_review_budget
fig2_review_budget.make(show=True)

## Table I — subsets and splits

In [ ]:
from analysis import table1_splits
t1 = table1_splits.make(show=True)

## Table II — main results with 95% CIs

In [ ]:
from analysis import table2_main_results
t2 = table2_main_results.make(show=True)

## Table III — late fusion at 100k

In [ ]:
from analysis import table3_late_fusion
t3 = table3_late_fusion.make(show=True)

## Table IV — review budget and TREC07

In [ ]:
from analysis import table4_review_budget
t4 = table4_review_budget.make(show=True)

## Integrity check
SHA-256 of every saved-result file read and every output written; re-hashing must match.

In [ ]:
print(C.write_manifest().read_text())
!python -m analysis.verify_manifest

## Download everything

In [ ]:
!cd {REPO} && zip -qr /content/outputs.zip outputs
from google.colab import files
files.download('/content/outputs.zip')